<div style="text-align: center; background-color: #5A96E3; font-family: 'Trebuchet MS', Arial, sans-serif; color: white; padding: 20px; font-size: 40px; font-weight: bold; border-radius: 0 0 0 0; box-shadow: 0px 6px 8px rgba(0, 0, 0, 0.2);">
  Stage 03 - Extract candidate cv 📌
</div>

## I. Import libraries

In [ ]:
import pandas as pd
import fitz
import os
import json
import re
import uuid
from langchain_qdrant import Qdrant
import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.models import Filter, FieldCondition, MatchValue
# from qdrant_client.models import t, VectorParams, Distance
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import GPT4AllEmbeddings
from langchain.schema import HumanMessage
from langchain.schema import Document



## II. Config environment

In [ ]:
os.environ["DEEPSEEK_API_KEY"] = ""
llm = ChatDeepSeek(model="deepseek-chat")

## II. Extracting cv content from pdf format

Extrating raw text content in a CV.

In [11]:
def extract_text_from_pdf(file_path: str) -> str:
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()


Extract each field using a LLM model.

Define a LLM and a embedding model.

Define a prompt template for extrating

In [13]:
prompt_template = """
Extract the following candidate information fields from the CV content (as plain text) below in the exact JSON format:
{{
"full_name": "...",
"email": "...",
"phone": "...",
"job_title": "...",
"education": [
    {{
    "degree": "...",
    "university": "...",
    "start_year": ...,
    "end_year": ...
    }}
],
"experience": [
    {{
    "job_title": "...",
    "company": "...",
    "start_date": "...",
    "end_date": "...",
    "description": "..."
    }}
],

"skills": ["...", "..."],
"certifications": [
    {{
    "certificate_name": "...",
    "organization": "..."
    }}
],
"languages": ["...", "..."]
}}

Only include **real work experience** (e.g. internships, jobs at companies, freelance work) in the "experience" field.  
**Do not include personal, academic, or side projects** in the experience section.

Only return the JSON content. Do not include any explanation.  
If any field cannot be found, set it to null or empty array.

CV content:
{text}
"""


In [14]:
def extract_info(text: str) -> dict:
    prompt = prompt_template.format(text=text)
    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)
    raw_content = response.content

    
    cleaned_data = re.sub(r"^```json\s*|\s*```$", "", raw_content.strip(), flags=re.MULTILINE)
    # candidate_info = json.loads(response.content)
    
    try:
        candidate_info = json.loads(cleaned_data)

        # Lọc experience: bỏ các mục có company = None hoặc ""
        if "experience" in candidate_info and isinstance(candidate_info["experience"], list):
            filtered_exp = []
            for exp in candidate_info["experience"]:
                company = exp.get("company")
                if company not in [None, ""]:
                    filtered_exp.append(exp)
            candidate_info["experience"] = filtered_exp

    except Exception as e:
        print(f"Error parsing JSON: {e}\nLLM output: {cleaned_data}")
        candidate_info = {}
    return candidate_info

    

In [ ]:
def make_id(text: str) -> int:
    # Sinh int id từ string bằng hash
    return int(hashlib.md5(text.encode()).hexdigest(), 16) % (10**12)

In [18]:
def process_cvs(input_dir: str, vector_db, embedding_model, collection_name: str, limit: int = 5):
    """
    Đọc các file PDF trong thư mục input_dir, trích xuất thông tin (skills, experiences)
    và lưu vào vector database.
    """

    # Lấy danh sách file PDF
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    pdf_files = pdf_files[:limit]

    for filename in pdf_files:
        file_path = os.path.join(input_dir, filename)
        print(f"Processing {file_path}...")

        # Trích xuất text từ PDF
        text = extract_text_from_pdf(file_path)
        if not text:
            print(f"⚠️ No text extracted from {filename}, skipping.")
            continue

        # Trích xuất thông tin từ LLM
        info = extract_info(text) or {}
        info["source_file"] = filename

        # Chuẩn hóa skills
        skills = info.get("skills", [])
        if isinstance(skills, str):
            skills = [skills]
        elif skills is None:
            skills = []

        # Chuẩn hóa experiences
        experiences = info.get("experience") or []

        # Gom tất cả point vào list
        points = []

        # Xử lý skills
        for skill in skills:
            vector = embedding_model.embed_query(skill)
            points.append(
                PointStruct(
                    id=make_id(f"skill-{filename}-{skill}-{uuid.uuid4().hex[:8]}"),
                    vector=vector,
                    payload={
                        "type": "skill",
                        "skill": skill,
                        "job_title": info.get("job_title"),
                        "source_file": filename,
                        "candidate_name": info.get("full_name"),
                    },
                )
            )

        # Xử lý experiences
        for i, exp in enumerate(experiences):
            exp_text = f"{exp.get('job_title', '')} at {exp.get('company', '')} ({exp.get('start_date', '')} - {exp.get('end_date', '')}) {exp.get('description', '')}"
            vector = embedding_model.embed_query(exp_text)
            points.append(
                PointStruct(
                    id=make_id(f"exp-{filename}-{exp.get('company', 'unknown')}-{i}-{uuid.uuid4().hex[:8]}"),
                    vector=vector,
                    payload={
                        "type": "experience",
                        "experience": exp_text,
                        "experience_detail": exp,  # lưu cả dict gốc
                        "job_title": info.get("job_title"),
                        "source_file": filename,
                        "candidate_name": info.get("full_name"),
                    },
                )
            )

        # Upsert tất cả points của file này trong 1 lần
        if points:
            vector_db.upsert(
                collection_name=collection_name,
                points=points
            )
            print(f"✅ Inserted {len(points)} points from {filename}")
        else:
            print(f"⚠️ No skills/experiences extracted from {filename}")


In [ ]:
collection_name = "candidates"
embedding_model = GPT4AllEmbeddings()

client = QdrantClient(path="../qdrant_new_db")
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22272\1622018923.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [31]:
process_cvs("../../../raw/cvs", client, embedding_model, collection_name, 50)

Processing ../../../raw/cvs\01.pdf...
✅ Inserted 52 points from 01.pdf
Processing ../../../raw/cvs\02.pdf...
✅ Inserted 35 points from 02.pdf
Processing ../../../raw/cvs\03.pdf...
✅ Inserted 34 points from 03.pdf
Processing ../../../raw/cvs\04.pdf...
✅ Inserted 15 points from 04.pdf
Processing ../../../raw/cvs\05.pdf...
✅ Inserted 6 points from 05.pdf
Processing ../../../raw/cvs\06.pdf...
✅ Inserted 8 points from 06.pdf
Processing ../../../raw/cvs\07.pdf...
✅ Inserted 20 points from 07.pdf
Processing ../../../raw/cvs\08.pdf...
✅ Inserted 33 points from 08.pdf
Processing ../../../raw/cvs\09.pdf...
✅ Inserted 10 points from 09.pdf
Processing ../../../raw/cvs\10.pdf...
✅ Inserted 17 points from 10.pdf
Processing ../../../raw/cvs\11.pdf...
✅ Inserted 30 points from 11.pdf
Processing ../../../raw/cvs\12.pdf...
✅ Inserted 31 points from 12.pdf
Processing ../../../raw/cvs\13.pdf...
✅ Inserted 31 points from 13.pdf
Processing ../../../raw/cvs\14.pdf...
✅ Inserted 60 points from 14.pdf
Processi

In [ ]:
client.close()

In [39]:
query_text = "Find all candidates with Python skill"
query_vector = embedding_model.embed_query(query_text)

search_res = client.search(
    collection_name="candidates",
    query_vector=query_vector,
    query_filter=Filter(
        must=[
            FieldCondition(key="type", match=MatchValue(value="skill"))  # chỉ lấy skill
        ]
    )
)

print("=== Candidates with Python skill (semantic search) ===")
for r in search_res:
    print(f"Score: {r.score:.4f}")
    print(f"{r.payload.get('candidate_name')} - {r.payload.get('skill')}")

client.close()

=== Candidates with Python skill (semantic search) ===
Score: 0.3831
Ayush Jain - Python
Score: 0.3831
RIYA CHACKO - Python
Score: 0.3831
Jay Beaton - Python
Score: 0.3831
Dale-Kurt Murray - Python
Score: 0.3831
Jake Mofa - Python
Score: 0.3831
Jason Winnebeck - Python
Score: 0.3831
Richard Peres - Python
Score: 0.3831
George Claireaux - Python
Score: 0.3831
Rajesh Kumar - Python
Score: 0.3831
Dale-Kurt Murray - Python


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22272\1028607644.py:4: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_res = client.search(


In [27]:
query_text = "Experience at Verana Health"
query_vector = embedding_model.embed_query(query_text)

# Search với filter
search_res = client.search(
    collection_name="candidates",
    query_vector=query_vector,
    limit=5,
    query_filter=Filter(
        must=[
            FieldCondition(key="candidate_name", match=MatchValue(value="Alexander Antonison")),
            FieldCondition(key="type", match=MatchValue(value="experience")),
        ]
    )
)

print("=== Work Experience at Verana Health (Alexander Antonison) ===")
for r in search_res:
    print(f"Score: {r.score:.4f}")
    print(r.payload.get("experience"))
    print("-----")

=== Work Experience at Verana Health (Alexander Antonison) ===
Score: 0.5660
Data Engineer at Verana Health (March 2021 - November 2021) Worked on building an ingestion platform that pulls in data from Electronic Health Record (EHR) systems and processes them into a common data model. Built PySpark Data Quality modules, AWS Glue PySpark jobs, and application templates.
-----
Score: 0.3692
Data Solutions Consultant at Antonison Consulting Group (June 2020 - Present) Help clients by providing solutions that foster growth and efficiency through Data Engineering and Architecture services, working with various clients including MTSU Data Science Institute, Concert Genetics, Verana Health, and Picknic.
-----
Score: 0.1809
Principal Machine Learning Engineer at Stratasan (October 2019 - February 2021) Started as Data Services Manager leading Data Engineering and Data Science teams, then transitioned to Principal Machine Learning Engineer focusing on scaling data products and supporting machin

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_22272\626721648.py:5: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_res = client.search(


In [8]:
import os
input_dir = "../../../raw/cvs"
limit = 50
pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
pdf_files = pdf_files[:limit]
print(pdf_files)

['01.pdf', '02.pdf', '03.pdf', '04.pdf', '05.pdf', '06.pdf', '07.pdf', '08.pdf', '09.pdf', '10.pdf', '11.pdf', '12.pdf', '13.pdf', '14.pdf', '15.pdf', '16.pdf', '17.pdf', '18.pdf', '19.pdf', '20.pdf', '21.pdf', '22.pdf', '23.pdf', '24.pdf', '25.pdf', '26.pdf', '27.pdf', '28.pdf', '29.pdf', '30.pdf', '31.pdf', '32.pdf', '33.pdf', '34.pdf', '35.pdf', '36.pdf', '37.pdf', '38.pdf', '39.pdf', '40.pdf', '41.pdf', '42.pdf', '43.pdf', '44.pdf', '45.pdf', '46.pdf', '47.pdf', '48.pdf', '49.pdf', '50.pdf']


## III. Add to Qdrant vector db

In [3]:
collection_name = "candidates"
embedding_model = GPT4AllEmbeddings()

# client = QdrantClient(path="../qdrant_initial_db")
# client.recreate_collection(
#     collection_name=collection_name,
#     vectors_config=VectorParams(size=384, distance=Distance.COSINE)
# )

In [ ]:
# with open("../cv_database/json_file/candidates.json", "r", encoding="utf-8") as f:
#     candidates = json.load(f)

# points = []
# for cand in candidates:
#     name = cand.get("full_name", "")
#     email = cand.get("email", "")
#     skills = cand.get("skills", [])
#     exp_list = cand.get("experience", [])

#     if isinstance(skills, str):  
#         skills = [skills]   # ép thành list nếu là string
#     elif skills is None:
#         skills = []

#     # Chuẩn hóa text để embedding
#     exp_texts = [f"{e.get('job_title','')} at {e.get('company','')}" for e in exp_list]
#     exp_text = " | ".join(exp_texts)

#     text_to_embed = f"Name: {name}, Email: {email}, Skills: {', '.join(skills)}, Experience: {exp_text}"

#     #vector = embedding_model.encode(text_to_embed).tolist()
#     vector = embedding_model.embed_query(text_to_embed)
#     points.append(
#         PointStruct(
#             id=str(uuid.uuid4()),  # cải thiện Indexing
#             vector=vector,
#             payload={ #thêm metadata
#                 "name": name,
#                 "email": email,
#                 "skills": skills,
#                 "experience": exp_text,
#                 "text": text_to_embed
#             }, #tăng độ chi tiết dữ liệu được index
#         )
#     )

# # Insert vào Qdrant 
# client.upsert(collection_name=collection_name, points=points)

# print(f"Inserted {len(points)} candidates into Qdrant 🚀")


Inserted 33 candidates into Qdrant 🚀


In [ ]:
with open("../cv_database/json_file/candidates.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)

all_docs = []
for cand in candidates:
    name = cand.get("full_name", "")
    email = cand.get("email", "")
    skills = cand.get("skills", [])
    exp_list = cand.get("experience", [])

    if isinstance(skills, str):  
        skills = [skills]   # ép thành list nếu là string
    elif skills is None:
        skills = []

    #  chuẩn hóa experience thành chuỗi, Enhancing data granularity (tăng độ chi tiết dữ liệu)
    exp_texts = [f"{e.get('job_title','')} at {e.get('company','')}" for e in exp_list]
    exp_text = " | ".join(exp_texts)

    text_to_embed = f"Name: {name}, Email: {email}, Skills: {', '.join(skills)}, Experience: {exp_text}"

    all_docs.append(
        Document(
            page_content=text_to_embed,
            metadata={  #thêm metadata
                "name": name,
                "email": email,
                "skills": skills,
                "experience": exp_text
            }
        )
    )
    
# collection_name = "candidates"
# embedding_model = GPT4AllEmbeddings()
    
vectorstore = Qdrant.from_documents(
    all_docs,
    embedding=embedding_model,
    collection_name=collection_name,
    path="../qdrant_db")    


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k":6})
results = retriever.get_relevant_documents("6 Frontend React.js developers")

for doc in results:
    print(doc.metadata)
    print(doc.page_content)

{'name': 'Obi Nwokogba', 'email': 'obi.nwokogba@gmail.com', 'skills': ['JavaScript', 'TypeScript', 'Python', 'CSS3', 'HTML5', 'Java', 'PHP', 'SQL', 'Sass', 'MQL', 'Angular', 'React', 'React Native', 'HTMX', 'Express', 'NestJS', 'MongoDB', 'NodeJS', 'Responsive Design', 'Android App Development', 'UI design', 'Database Architecture', 'Git', 'Data Structures', 'Version Repository', 'Web APIs', 'Data Visualization', 'Agile Methodology', 'Front-End Web Development', 'Full-Stack Web Development', 'MySQL', 'Bootstrap', 'ERDs', 'Graphic Design', 'Algorithmic Trading', 'Financial Markets'], 'experience': 'Frontend Engineer at Madison Logic Inc. | Full Stack Engineer, Graphic Designer at Pregen Inc.', '_id': '438a7579820048b7b961abc86ffa6849', '_collection_name': 'candidates'}
Name: Obi Nwokogba, Email: obi.nwokogba@gmail.com, Skills: JavaScript, TypeScript, Python, CSS3, HTML5, Java, PHP, SQL, Sass, MQL, Angular, React, React Native, HTMX, Express, NestJS, MongoDB, NodeJS, Responsive Design, A